# Train Neural Network with PyTorch


### 1. Load libraries

In [1]:
!pip install torch
!pip install torchvision
!pip install tqdm
!pip install matplotlib
!pip install --upgrade "numpy<2"

In [2]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import Compose, ToTensor, Lambda
from tqdm import tqdm

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def move_to_device(tensor: torch.Tensor, device: torch.device) -> torch.Tensor:
    return tensor.to(device)

Torch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130


### 2. Construct neural network

In [3]:
class DigitClassifier(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
        self.loss_function = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters())
        self.device = device
        self.to(self.device)

    def forward(self, input_images: torch.Tensor) -> torch.Tensor:
        return self.network(input_images)

    def predict(self, input_images: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            logits = self.forward(input_images)
            predicted_labels = torch.argmax(logits, dim=-1)
            return predicted_labels

    def train_on_batch(self, batch_images: torch.Tensor, batch_labels: torch.Tensor) -> float:
        self.optimizer.zero_grad()
        logits = self.forward(batch_images)
        loss = self.loss_function(logits, batch_labels)
        loss.backward()
        self.optimizer.step()
        return loss.item()


### 3. Prepare dataset

In [4]:
device = get_device()
print("Using device:", device)

mnist_transform = Compose([
    ToTensor(),
    Lambda(lambda image: image.view(784))
])

train_dataset = MNIST(
    root="./",
    train=True,
    download=True,
    transform=mnist_transform
)

test_dataset = MNIST(
    root="./",
    train=False,
    download=True,
    transform=mnist_transform
)

print("First training sample shape:", train_dataset[0][0].shape)
print("First training label:", train_dataset[0][1])

Using device: cpu
First training sample shape: torch.Size([784])
First training label: 5


### 4. Train network

In [5]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

model = DigitClassifier(device=device)

num_epochs = 5

for epoch_index in range(num_epochs):
    model.train()
    total_epoch_loss = 0.0

    for batch_images_cpu, batch_labels_cpu in tqdm(train_loader, desc=f"Training epoch {epoch_index + 1}/{num_epochs}"):
        batch_images = move_to_device(batch_images_cpu, device)
        batch_labels = move_to_device(batch_labels_cpu, device)

        batch_loss = model.train_on_batch(batch_images, batch_labels)
        total_epoch_loss += batch_loss

    average_epoch_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch {epoch_index + 1}: average loss = {average_epoch_loss:.4f}")

model.eval()
correct_predictions = 0
total_predictions = 0

Training epoch 1/5: 100%|██████████| 3750/3750 [00:25<00:00, 144.98it/s]


Epoch 1: average loss = 0.1970


Training epoch 2/5: 100%|██████████| 3750/3750 [00:30<00:00, 123.08it/s]


Epoch 2: average loss = 0.0872


Training epoch 3/5: 100%|██████████| 3750/3750 [00:32<00:00, 115.24it/s]


Epoch 3: average loss = 0.0623


Training epoch 4/5: 100%|██████████| 3750/3750 [00:28<00:00, 131.48it/s]


Epoch 4: average loss = 0.0492


Training epoch 5/5: 100%|██████████| 3750/3750 [00:25<00:00, 149.54it/s]

Epoch 5: average loss = 0.0417


### 5. Use trained network

In [6]:
for batch_images_cpu, batch_labels_cpu in test_loader:
    batch_images = move_to_device(batch_images_cpu, device)
    batch_labels = move_to_device(batch_labels_cpu, device)

    predicted_labels = model.predict(batch_images)
    correct_predictions += (predicted_labels == batch_labels).sum().item()
    total_predictions += batch_labels.size(0)

test_accuracy = correct_predictions / total_predictions
print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.9799
